* In this assignment you will be using the entire transformer architecture for a translation task.
* we will just be using one encoder layer and one decoder layer
* You can copy the encoder and decoder modules from the previous assignments. You are going to translate a few sentences from **English to Tamil**
  * Source language: English
  * Target language: Tamil

* You may experiment with a target language of your choice for checking the impelementation. (You may use google translate for that)

* We need to install torchdata and torchtext (which take about 3 minutes to finish installing) for tokenizing the text.
* We already defined useful functions for the tokenization of texts




In [1]:
!pip install torchdata==0.6.0 # to be compatible with torch 2.0
!pip install portalocker==2.0.0
!pip install -U torchtext==0.15.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

* Let's import all required libraries

In [336]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

#text lib
import torchtext

# tokenizer
from torchtext.data.utils import get_tokenizer

#build vocabulary
from torchtext.vocab import vocab
from torchtext.vocab import build_vocab_from_iterator

# get input_ids (numericalization)
from torchtext.transforms import VocabTransform, LabelToIndex

# get embeddings
from torch.nn import Embedding

from  pprint import pprint
from yaml import safe_load
import copy
import numpy as np
import requests
import math

# Preparing Data

* Source and target text

In [337]:
src_text = """The most famous ruler of ancient India was Emperor Ashoka.
It was during his period that Buddhism spread to different parts of Asia.
Ashoka gave up war after seeing many people grieving death after the Kalinga war.
He embraced Buddhism and then devoted his life to spread the message of peace and dharma.
His service for the cause of public good was exemplary.
He was the first ruler to give up war after victory.
He was the first to build hospitals for animals.
He was the first to lay roads."""

In [338]:
tar_text = """பண்டைய இந்திய அரசர்களில் பேரும் புகழும் பெற்ற அரசர் அசோகர் ஆவார்.
இவரது ஆட்சியில் தான் புத்த மதம் ஆசியாவின் பல்வேறு பகுதிகளுக்குப் பரவியது.
கலிங்கப் போருக்குப் பின் பல உயிர்கள் மடிவதைக் கண்டு வருந்தி, போர் தொடுப்பதைக் கைவிட்டார்.
அதற்குப் பிறகு புத்த சமயத்தைத் தழுவி, அமைதியையும் அறத்தையும் பரப்புவதற்காகத் தன் வாழ்வையே அர்ப்பணித்தார்.
பொதுமக்களுக்கு அவர் ஆற்றிய சேவை முன் மாதிரியாக விளங்கியது.
வெற்றிக்குப் பின் போரைத் துறந்த முதல் அரசர் அசோகர்தான்.
உலகிலேயே முதன்முதலாக விலங்குகளுக்கும் தனியே மருத்துவமனை அமைத்துத் தந்தவரும் அசோகரே ஆவார்.
 இன்றும் அவர் உருவாக்கிய சாலைகளை நாம் பயன்படுத்திக்கொண்டு இருக்கிறோம்."""

* Tokenize and build vocabulary using a simple tokenization algorithm

In [339]:
# do not edit this cell
def seq_len(seq):
  return len(seq.strip('').split(' '))

# check the maximum length of the src and target seq to decide the context length of encdoer and decoder
src_raw_seq = src_text.strip('').split('\n')
src_max_seq_len =max(list(map(seq_len,src_raw_seq)))
print('Source max_seq_length:  ',src_max_seq_len)


tar_raw_seq = tar_text.strip('').split('\n')
tar_max_seq_len =max(list(map(seq_len,tar_raw_seq)))
print('Target max_seq_length: ',tar_max_seq_len)

Source max_seq_length:   16
Target max_seq_length:  11


* We encourage you to go through the code given below to understand the typical functionalities of Tokenizer object (If you want, you can skip)

In [340]:
# do not edit this cell
class Tokenizer(object):

  def __init__(self,text):
    self.text = text
    self.word_tokenizer = self.word_tokenizer
    self.vocab_size = None
    self.vocab = None

  @staticmethod
  def word_tokenizer(seq):
    return seq.strip('').split(' ')

  def get_tokens(self):
    for sentence in self.text.strip().split('\n'):
      yield self.word_tokenizer(sentence)

  def build_vocab(self):
    self.vocab = build_vocab_from_iterator(self.get_tokens(),
                                  min_freq=1,specials=['<pad>','<start>','<end>','<unk>'])
    self.vocab.set_default_index(self.vocab['<unk>']) # index of OOV
    self.vocab_size = len(self.vocab)
    return self.vocab

  def encode(self,sentence):
    v = self.build_vocab()
    vt = VocabTransform(v)
    token_ids = vt(self.word_tokenizer(sentence))
    # add special tokens
    token_ids.insert(0,v.vocab.get_stoi()['<start>'])
    token_ids.append(v.vocab.get_stoi()['<end>']) # <end>:2
    return torch.tensor(token_ids,dtype=torch.int64)

  def decode(self,ids):
    v = self.build_vocab()
    list_ids = ids.tolist()
    tokens = [v.vocab.get_itos()[id] for id in list_ids]
    return ' '.join(tokens)

  def encode_batch(self,batch_size,max_seq_len):
    batch_data = torch.zeros(size=(batch_size,max_seq_len+2)) # +2 for special tokens
    for i,sentence in enumerate(self.text.strip('').split('\n')):
      token_ids = self.encode(sentence)
      batch_data[i,0:len(token_ids)] = token_ids
    return batch_data.type(dtype=torch.int64)



* It is always go to check the implementation

In [341]:
batch_size = 8

In [342]:
# you can play with this
src_tokenizer = Tokenizer(src_text)
print(src_tokenizer.encode('The most famous ruler of ancient India was Emperor Ashoka.'))
print(src_tokenizer.encode_batch(batch_size,src_max_seq_len))

tensor([ 1, 27, 49, 39, 15,  8, 28, 24,  5, 22, 20,  2])
tensor([[ 1, 27, 49, 39, 15,  8, 28, 24,  5, 22, 20,  2,  0,  0,  0,  0,  0,  0],
        [ 1, 25,  5, 36, 14, 53, 58, 11, 16,  6, 35, 50,  8, 21,  2,  0,  0,  0],
        [ 1, 19, 40, 17, 18,  9, 56, 47, 52, 43, 32,  9,  4, 26, 61,  2,  0,  0],
        [ 1,  7, 37, 11, 12, 59, 33, 14, 46,  6, 16,  4, 48,  8, 51, 12, 34,  2],
        [ 1, 23, 57, 13,  4, 31,  8, 54, 42,  5, 38,  2,  0,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10, 15,  6, 41, 17, 18,  9, 60,  2,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10,  6, 30, 44, 13, 29,  2,  0,  0,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10,  6, 45, 55,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0]])


In [343]:
# you can play with this
tar_tokenizer = Tokenizer(tar_text)
print(tar_tokenizer.encode('பண்டைய இந்திய அரசர்களில் பேரும் புகழும் பெற்ற அரசர் அசோகர் ஆவார்.'))
print(tar_tokenizer.encode_batch(batch_size,tar_max_seq_len))

tensor([ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2])
tensor([[ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2,  0,  0],
        [ 1, 25, 20, 39,  8, 59, 19, 49, 43, 47,  2,  0,  0],
        [ 1, 30, 55,  7, 48, 26, 58, 29, 65, 57, 41, 31,  2],
        [ 1, 13, 50,  8, 32, 38, 14, 18, 46, 37, 66, 17,  2],
        [ 1, 54,  5, 21, 34, 64, 61, 68,  2,  0,  0,  0,  0],
        [ 1, 69,  7, 56, 40, 63,  4, 12,  2,  0,  0,  0,  0],
        [ 1, 28, 62, 67, 36, 60, 15, 35, 10,  6,  2,  0,  0],
        [ 1,  9, 23,  5, 27, 33, 42, 45, 24,  2,  0,  0,  0]])


* Let's load the token ids of the words in the sentences of source and target languages

In [344]:
# do not edit this cell
x = src_tokenizer.encode_batch(batch_size,src_max_seq_len)
y = tar_tokenizer.encode_batch(batch_size,tar_max_seq_len)

* we have appended zeros to sentences that are shorter than max-seq-len
* We have to ignore computing loss over those padded tokens
* You have to take care of that in the cell below

In [345]:
x,y

(tensor([[ 1, 27, 49, 39, 15,  8, 28, 24,  5, 22, 20,  2,  0,  0,  0,  0,  0,  0],
         [ 1, 25,  5, 36, 14, 53, 58, 11, 16,  6, 35, 50,  8, 21,  2,  0,  0,  0],
         [ 1, 19, 40, 17, 18,  9, 56, 47, 52, 43, 32,  9,  4, 26, 61,  2,  0,  0],
         [ 1,  7, 37, 11, 12, 59, 33, 14, 46,  6, 16,  4, 48,  8, 51, 12, 34,  2],
         [ 1, 23, 57, 13,  4, 31,  8, 54, 42,  5, 38,  2,  0,  0,  0,  0,  0,  0],
         [ 1,  7,  5,  4, 10, 15,  6, 41, 17, 18,  9, 60,  2,  0,  0,  0,  0,  0],
         [ 1,  7,  5,  4, 10,  6, 30, 44, 13, 29,  2,  0,  0,  0,  0,  0,  0,  0],
         [ 1,  7,  5,  4, 10,  6, 45, 55,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0]]),
 tensor([[ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2,  0,  0],
         [ 1, 25, 20, 39,  8, 59, 19, 49, 43, 47,  2,  0,  0],
         [ 1, 30, 55,  7, 48, 26, 58, 29, 65, 57, 41, 31,  2],
         [ 1, 13, 50,  8, 32, 38, 14, 18, 46, 37, 66, 17,  2],
         [ 1, 54,  5, 21, 34, 64, 61, 68,  2,  0,  0,  0,  0],
         [ 1, 69,  

In [346]:
# your code goes here

label = y
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

* Define the context lengths for encoder and decoder

In [347]:
# do not edit this cell
enc_ctxt_len = src_max_seq_len+2
dec_ctxt_len = tar_max_seq_len+2

# Load configuration file

In [348]:
# do not edit this cell
config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/enc_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 10},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [349]:
# do not edit this cell
src_vocab_size =src_tokenizer.vocab_size
batch_size = x.shape[0]
embed_dim = config['input']['embed_dim']

In [350]:
src_vocab_size

62

In [351]:
x.max().item()

61

In [352]:
# do not edit this cell
dq = torch.tensor(config['model']['dq'])
dk = torch.tensor(config['model']['dk'])
dv = torch.tensor(config['model']['dv'])
dmodel = embed_dim
heads = torch.tensor(config['model']['n_heads'])
d_ff = config['model']['d_ff']

In [353]:
# do not edit this cell
config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/dec_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 12},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [354]:
# do not edit this cell
tar_vocab_size = tar_tokenizer.vocab_size

# Encoder

 * You can copy paste the required code from the previous assignments

In [355]:
b=True
class MHA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHA,self).__init__()
    # your code goes here
    self.dmodel = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv
    self.heads = heads


    # init QKV and O
    self.W_q = nn.Linear(dmodel, dq * heads, bias=b)
    self.W_k = nn.Linear(dmodel, dk * heads, bias=b)
    self.W_v = nn.Linear(dmodel, dv * heads, bias=b)
    self.W_o = nn.Linear(dv * heads, dmodel, bias=b)




    torch.manual_seed(43)
    nn.init.normal_(self.W_q.weight)
    if b:
        nn.init.normal_(self.W_q.bias)

    torch.manual_seed(44)
    nn.init.normal_(self.W_k.weight)
    if b:
        nn.init.normal_(self.W_k.bias)

    torch.manual_seed(45)
    nn.init.normal_(self.W_v.weight)
    if b:
        nn.init.normal_(self.W_v.bias)

    torch.manual_seed(46)
    nn.init.normal_(self.W_o.weight)
    if b:
        nn.init.normal_(self.W_o.bias)

  # your method definitions go here (if you want to)

  def forward(self,H=None):
    '''
    Input: Size [BSxTxdmodel]
    Output: Size[BSxTxdmodel]
    '''
    # your code goes here

    # print(H.shape,'mha input')

    BS, T, _ = H.size()

    # H *  Q, K, V
    Q = self.W_q(H)
    K = self.W_k(H)
    V = self.W_v(H)

    # print(V.shape,'qkv shape')


    # multiple heads
    Q = Q.view(BS, T, self.heads, self.dq).transpose(1, 2)
    K = K.view(BS, T, self.heads, self.dk).transpose(1, 2)
    V = V.view(BS, T, self.heads, self.dv).transpose(1, 2)
    # print(V.shape,'multi head qkv shape')


    #  attention
    scores = torch.matmul(Q, K.transpose(-2,-1)) / (self.dk ** 0.5)
    # print(scores.shape,'scores shape')
    attn_weights = F.softmax(scores, dim=-1)
    # print(attn_weights.shape,'attention weights')
    context = torch.matmul(attn_weights, V)
    # print(context.shape,'context shape')

    # concat
    context = context.transpose(1, 2).contiguous().view(BS, T, self.dv*self.heads)

    # output
    out = self.W_o(context)
    return out

class Prediction(nn.Module):

  def __init__(self,dmodel,vocab_size):
    super(Prediction,self).__init__()
    # your code goes here

    self.proj = nn.Linear(dmodel, vocab_size,bias=b)



    torch.manual_seed(49)
    nn.init.normal_(self.proj.weight)
    if b:
      nn.init.normal_(self.proj.bias)


  def forward(self,representations):
    '''
    input: size [bsxTxdmodel]
    output: size [bsxTxvocab_size]
    Note: Do not apply the softmax. Just return the output of linear transformation
    '''
    out = self.proj(representations)
    return out

class EncoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads):
    super(EncoderLayer,self).__init__()
    self.mha = MHA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mha = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,x):


    # print(x.shape,'encoder input')
    attn_out = self.mha(x)
    # print('mha output created')


    x = self.layer_norm_mha(x + attn_out)
    ffn_out = self.ffn(x)
    out = self.layer_norm_ffn(x + ffn_out)


    return out
class Encoder(nn.Module):

  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=2):
    super(Encoder,self).__init__()
    # self.embed_weights = nn.Embedding(vocab_size, dmodel)
    self.enc_layers = nn.ModuleList([
        EncoderLayer(dmodel, dq, dk, dv, d_ff, heads)
        for _ in range(num_layers)
    ])
    self.output_layer = Prediction(dmodel, vocab_size)

  def get_embeddings(self, x):
      return self.embed_weights(x)

  def forward(self,x):

    out = x


    for layer in self.enc_layers:
        out = layer(out)

    # out = self.output_layer(out)
    return out



class FFN(nn.Module):
  def __init__(self,dmodel,d_ff,layer=0):
    super(FFN,self).__init__()
    #your code goes here
    #layers
    self.fc1 = nn.Linear(dmodel, d_ff, bias=b)
    self.fc2 = nn.Linear(d_ff, dmodel, bias=b)

    # init
    torch.manual_seed(47)
    nn.init.normal_(self.fc1.weight)
    if b:
        nn.init.normal_(self.fc1.bias)

    torch.manual_seed(48)
    nn.init.normal_(self.fc2.weight)
    if b:
        nn.init.normal_(self.fc2.bias)


  def forward(self,x):
    '''
    input: size [BSxTxdmodel]
    output: size [BSxTxdmodel]
    '''
    #your code goes here
    out = self.fc1(x)
    out = F.relu(out)
    out = self.fc2(out)
    return out

# Decoder

In [356]:
class MHCA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHCA,self).__init__()
    self.dmodel = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv
    self.heads = heads


    self.W_q = nn.Linear(dmodel, dq * heads, bias=b)
    self.W_k = nn.Linear(dmodel, dk * heads, bias=b)
    self.W_v = nn.Linear(dmodel, dv * heads, bias=b)
    self.W_o = nn.Linear(dv * heads, dmodel, bias=b)

    torch.manual_seed(43)
    nn.init.normal_(self.W_q.weight)
    if b:
        nn.init.normal_(self.W_q.bias)

    torch.manual_seed(44)
    nn.init.normal_(self.W_k.weight)
    if b:
        nn.init.normal_(self.W_k.bias)

    torch.manual_seed(45)
    nn.init.normal_(self.W_v.weight)
    if b:
        nn.init.normal_(self.W_v.bias)

    torch.manual_seed(46)
    nn.init.normal_(self.W_o.weight)
    if b:
        nn.init.normal_(self.W_o.bias)



     # your method definitions go here (if you want to)

  def forward(self,Q_input,H=None):
    BS, T_dec, _ = Q_input.size()

    BS, T_enc, _ = H.size()
    Q = self.W_q(Q_input)
    K = self.W_k(H)
    V = self.W_v(H)


    Q = Q.view(BS, T_dec, self.heads, self.dq).transpose(1, 2)
    K = K.view(BS, T_enc, self.heads, self.dk).transpose(1, 2)
    V = V.view(BS, T_enc, self.heads, self.dv).transpose(1, 2)


    # attention
    scores = torch.matmul(Q, K.transpose(-2,-1)) / (self.dk ** 0.5)

    # print(scores.shape,'scores shape')
    attn_weights = F.softmax(scores, dim=-1)
    # print(attn_weights.shape,'attention weights')
    context = torch.matmul(attn_weights, V)
    # print(context.shape,'context shape')

    # concat heads
    context = context.transpose(1, 2).contiguous().view(BS, T_dec, self.dv*self.heads)

    out = self.W_o(context)


    return out
b=False
class MHMA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads,mask=None):
    super(MHMA,self).__init__()
    # your code goes here
    self.dmodel = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv
    self.heads = heads


    self.W_q = nn.Linear(dmodel, dq * heads, bias=b)
    self.W_k = nn.Linear(dmodel, dk * heads, bias=b)
    self.W_v = nn.Linear(dmodel, dv * heads, bias=b)
    self.W_o = nn.Linear(dv * heads, dmodel, bias=b)

    torch.manual_seed(43)
    nn.init.normal_(self.W_q.weight)
    if b:
        nn.init.normal_(self.W_q.bias)

    torch.manual_seed(44)
    nn.init.normal_(self.W_k.weight)
    if b:
        nn.init.normal_(self.W_k.bias)

    torch.manual_seed(45)
    nn.init.normal_(self.W_v.weight)
    if b:
        nn.init.normal_(self.W_v.bias)

    torch.manual_seed(46)
    nn.init.normal_(self.W_o.weight)
    if b:
        nn.init.normal_(self.W_o.bias)



  def forward(self,H=None):
    # print('hhh')
    # implement forward method

    BS, T, _ = H.size()
    # print('hhh')
    # print(H.shape)
    Q = self.W_q(H)

    K = self.W_k(H)
    V = self.W_v(H)
    Q = Q.view(BS, T, self.heads, self.dq).transpose(1, 2)
    K = K.view(BS, T, self.heads, self.dk).transpose(1, 2)
    V = V.view(BS, T, self.heads, self.dv).transpose(1, 2)



    scores = torch.matmul(Q, K.transpose(-2,-1)) / (self.dk ** 0.5)
    mask = torch.tril(torch.ones(T, T, device=H.device)).unsqueeze(0).unsqueeze(0)
    scores = scores.masked_fill(mask == 0, float('-inf'))

    # print(scores.shape,'scores shape')
    attn_weights = F.softmax(scores, dim=-1)
    # print(attn_weights.shape,'attention weights')
    context = torch.matmul(attn_weights, V)
    # print(context.shape,'context shape')

    # concat heads
    context = context.transpose(1, 2).contiguous().view(BS, T, self.dv*self.heads)

    # output
    out = self.W_o(context)

    return out
class DecoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads,mask=None):
    super(DecoderLayer,self).__init__()
    self.mhma = MHMA(dmodel,dq,dk,dv,heads,mask=None)
    self.mhca = MHCA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mhma = torch.nn.LayerNorm(dmodel)
    self.layer_norm_mhca = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,x,enc_rep):
    # print(x.shape,'encoder input')
    attn_out = self.mhma(x)
    # print('mha output created')


    x = self.layer_norm_mhma(x + attn_out)


    c_attn_out = self.mhca(x,enc_rep)
    x = self.layer_norm_mhca(x + c_attn_out)


    # ffn out with layernorm
    ffn_out = self.ffn(x)
    out = self.layer_norm_ffn(x + ffn_out)

    return out
class Decoder(nn.Module):
  def __init__(self, vocab_size, dmodel, dq, dk, dv, d_ff, heads, mask, num_layers=1):
    super(Decoder, self).__init__()

    self.dec_layers = nn.ModuleList([
      DecoderLayer(dmodel, dq, dk, dv, d_ff, heads, mask=mask)
      for _ in range(num_layers)
    ])
    self.out = DOutputLayer(dmodel, vocab_size)

  def forward(self, enc_rep, tar_embedded):
    # print("Decoder input shape:", tar_embedded.shape)
    x = tar_embedded

    for layer in self.dec_layers:
      x = layer(x,enc_rep)

    out = self.out(x)
    return out


class FFN(nn.Module):
  def __init__(self,dmodel,d_ff):
    super(FFN,self).__init__()

    self.fc1 = nn.Linear(dmodel, d_ff, bias=b)
    self.fc2 = nn.Linear(d_ff, dmodel, bias=b)

    torch.manual_seed(47)
    nn.init.normal_(self.fc1.weight)
    if b:
        nn.init.normal_(self.fc1.bias)

    torch.manual_seed(48)
    nn.init.normal_(self.fc2.weight)
    if b:
        nn.init.normal_(self.fc2.bias)

  def forward(self,x):
    out = self.fc1(x)
    out = F.relu(out)
    out = self.fc2(out)
    return out

class Embed(nn.Module):

  def __init__(self,vocab_size,embed_dim):
    super(Embed,self).__init__()

    self.embed= nn.Embedding(vocab_size, embed_dim)
    torch.manual_seed(70)
    nn.init.normal_(self.embed.weight)


  def forward(self,x):
    out =x
    return out


class DOutputLayer(nn.Module):

  def __init__(self,dmodel,vocab_size):
    super(DOutputLayer,self).__init__()
    self.proj = nn.Linear(dmodel, vocab_size,bias=b)



    torch.manual_seed(49)
    nn.init.normal_(self.proj.weight)
    if b:
      nn.init.normal_(self.proj.bias)

  def forward(self,representations):
    out = self.proj(representations)
    return out

# Positional Embedding

 * You may take the code directly from any source.

In [357]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):

        x = x + self.pe[:, :x.size(1), :]
        return x

# Generate target mask

  * We will be passing the causal mask for the decoder layer as one of its arguments

In [358]:
mask = (torch.triu(torch.ones(dec_ctxt_len,dec_ctxt_len)) == 1).transpose(0,1)
mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
print(mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


# Transformer

In [359]:
class Transformer(nn.Module):
  def __init__(self, src_vocab_size, tar_vocab_size, src_seq_len, tar_seq_len,
               dmodel, dq, dk, dv, d_ff, heads, target_mask, num_layers=1):
    super(Transformer, self).__init__()

    self.src_embeddings = nn.Embedding(src_vocab_size, dmodel)
    self.tar_embeddings = nn.Embedding(tar_vocab_size, dmodel)

    self.pos_embeddings = PositionalEncoding(dmodel)

    self.encoder = Encoder(src_vocab_size, dmodel, dq, dk, dv, d_ff, heads, num_layers)
    self.decoder = Decoder(tar_vocab_size, dmodel, dq, dk, dv, d_ff, heads, target_mask, num_layers)


  def forward(self, src_token_ids, tar_token_ids):
      enc_input = self.pos_embeddings(self.src_embeddings(src_token_ids))
      dec_input = self.pos_embeddings(self.tar_embeddings(tar_token_ids))


      enc_out = self.encoder(self.pos_embeddings(self.src_embeddings(src_token_ids)))
      # print("Encoder output shape:", enc_out.shape)  # Should be [B, T, 32]


      enc_out = self.encoder(enc_input)
      # print("decoder input shape:", dec_input.shape)

      out = self.decoder(enc_out, dec_input)
      return out


In [360]:
model = Transformer(src_vocab_size,tar_vocab_size,enc_ctxt_len,dec_ctxt_len,dmodel,dq,dk,dv,d_ff,heads,mask)

In [361]:
criterion = loss_fn
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [362]:
def train(src_token_ids,tar_token_ids,labels,epochs=1000):
  loss_trace = []
  src_token_ids = src_token_ids.long()
  tar_token_ids = tar_token_ids.long()
  labels = labels.long()
  for epoch in range(epochs):
    # print("src_token_ids dtype:", src_token_ids.dtype)
    # print(src_token_ids.shape)

    out = model(src_token_ids,tar_token_ids)
    loss = criterion(out.view(-1, out.size(-1)), labels.view(-1))

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

In [363]:
train(x.long(),y.long(),label.long(),1000)

## Run the model AutoRegressively

In [364]:
def inference(token_ids, max_len=20, start_token_id=2, end_token_id=3, device='cpu'):
    model.eval()
    token_ids = token_ids.unsqueeze(0).to(device)

    with torch.no_grad():

        src_emb = model.src_embeddings(token_ids)
        src_emb = model.pos_embeddings(src_emb)
        enc_out = model.encoder(src_emb)



        generated = torch.tensor([[start_token_id]], dtype=torch.long, device=device)

        for _ in range(max_len):
            tar_emb = model.tar_embeddings(generated)
            tar_emb = model.pos_embeddings(tar_emb)
            dec_out = model.decoder(enc_out, tar_emb)

            next_token_logits = dec_out[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1).unsqueeze(1)
            generated = torch.cat([generated, next_token], dim=1)

            if next_token.item() == end_token_id:
                break

    return generated.squeeze(0)[1:]


* Modify the code below to suit your implementation
* Display the original and translated sentence (with all the spcial tokens)
* Note that, the second half of the second sentence is poorly translated
*  Same goes for 3rd and 4th sentence
* All other sentences are properly translated

In [365]:
for token_ids in x:
  print(src_tokenizer.decode(token_ids))
  print(tar_tokenizer.decode(inference(token_ids)))

<start> The most famous ruler of ancient India was Emperor Ashoka. <end> <pad> <pad> <pad> <pad> <pad> <pad>
அமைதியையும் பிறகு அசோகர் தந்தவரும் தழுவி, அர்ப்பணித்தார். தழுவி, அர்ப்பணித்தார். போருக்குப் மதம் பேரும் போர் புத்த பரப்புவதற்காகத் அமைதியையும் போருக்குப் பரப்புவதற்காகத் அரசர் அரசர் அரசர்
<start> It was during his period that Buddhism spread to different parts of Asia. <end> <pad> <pad> <pad>
அரசர்களில் பேரும் பரப்புவதற்காகத் <end> அரசர்களில் புத்த புத்த தன் <start> மருத்துவமனை தொடுப்பதைக் <end> <end> இந்திய துறந்த பரப்புவதற்காகத் <end> ஆட்சியில் <start> <end>
<start> Ashoka gave up war after seeing many people grieving death after the Kalinga war. <end> <pad> <pad>
முதல் <start> ஆற்றிய பின் விலங்குகளுக்கும் உயிர்கள் உயிர்கள் உயிர்கள் அரசர்களில் <end> தொடுப்பதைக் <start> விலங்குகளுக்கும் துறந்த உயிர்கள் தொடுப்பதைக் சாலைகளை <start> தொடுப்பதைக் <end>
<start> He embraced Buddhism and then devoted his life to spread the message of peace and dharma. <end>
முதல் அர்ப்பணித்தார். அமைதிய

In [366]:
for token_ids in x:
    print("SRC:", src_tokenizer.decode(token_ids))
    predicted = inference(token_ids)
    print("GEN:", tar_tokenizer.decode(predicted))


SRC: <start> The most famous ruler of ancient India was Emperor Ashoka. <end> <pad> <pad> <pad> <pad> <pad> <pad>
GEN: அமைதியையும் பிறகு அசோகர் தந்தவரும் தழுவி, அர்ப்பணித்தார். தழுவி, அர்ப்பணித்தார். போருக்குப் மதம் பேரும் போர் புத்த பரப்புவதற்காகத் அமைதியையும் போருக்குப் பரப்புவதற்காகத் அரசர் அரசர் அரசர்
SRC: <start> It was during his period that Buddhism spread to different parts of Asia. <end> <pad> <pad> <pad>
GEN: அரசர்களில் பேரும் பரப்புவதற்காகத் <end> அரசர்களில் புத்த புத்த தன் <start> மருத்துவமனை தொடுப்பதைக் <end> <end> இந்திய துறந்த பரப்புவதற்காகத் <end> ஆட்சியில் <start> <end>
SRC: <start> Ashoka gave up war after seeing many people grieving death after the Kalinga war. <end> <pad> <pad>
GEN: முதல் <start> ஆற்றிய பின் விலங்குகளுக்கும் உயிர்கள் உயிர்கள் உயிர்கள் அரசர்களில் <end> தொடுப்பதைக் <start> விலங்குகளுக்கும் துறந்த உயிர்கள் தொடுப்பதைக் சாலைகளை <start> தொடுப்பதைக் <end>
SRC: <start> He embraced Buddhism and then devoted his life to spread the message of peace and dharma.